In [ ]:
import pandas as pd

dm1 = pd.read_csv('../data/external/demographics/dem_exp1.csv')
dm2 = pd.read_csv('../data/external/demographics/dem_exp2.csv')
dm3 = pd.read_csv('../data/external/demographics/dem_exp3.csv')



df1 = pd.read_csv('../data/raw/exp2_2.csv')
df2 = pd.read_csv('../data/raw/exp366.csv')


df2 = df2[df2.expName.isin(['exp3'])]
df1 = df1[df1.expName.isin(['exp2', 'exp1'])]

df1 = df1.groupby('prolificID').filter(lambda x: len(x) >= 483)
df2 = df2.groupby('prolificID').filter(lambda x: len(x) >= 483)
# df = df.groupby('prolificID').filter(lambda x: len(x) <= 432)
# keep  prolificIDs that are more than 10 characters
df1 = df1[df1.prolificID.str.len() > 10]
df2 = df2[df2.prolificID.str.len() > 10]
# show prolificIDs 

In [21]:
dm1['prolificID'] = dm1['Participant id']
dm2['prolificID'] = dm2['Participant id']
dm3['prolificID'] = dm3['Participant id']

In [22]:
pids1 = df1.prolificID.unique()
pids2 = df2.prolificID.unique()

# check that pids1 are in dm1 and pids2 are in dm2
dm1 = dm1[(dm1.prolificID.isin(pids1)) | (dm1.prolificID.isin(pids2))]
dm2 = dm2[(dm2.prolificID.isin(pids1)) | (dm2.prolificID.isin(pids2))]
dm3 = dm3[(dm3.prolificID.isin(pids1)) | (dm3.prolificID.isin(pids2))]

In [23]:
# check dm1 and dm2 do not have the same prolificIDs
dm1 = dm1[~dm1.prolificID.isin(dm2.prolificID)]
dm2 = dm2[~dm2.prolificID.isin(dm1.prolificID)]
dm3 = dm3[~dm3.prolificID.isin(dm1.prolificID)]
dm3 = dm3[~dm3.prolificID.isin(dm2.prolificID)]


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = pd.read_csv('../data/raw/exp366.csv')
df2 = pd.read_csv('../data/raw/exp2_22.csv')
df3 = pd.read_csv('../data/raw/test/spaceprl(26).csv')

df = df[df.expName.isin(['exp1', 'exp2', 'exp3', ])]
df2 = df2[df2.expName.isin(['exp1', 'exp2'])]
df3 = df3[df3.expName.isin(['exp1B'])]
#concat df and df3
df = pd.concat([df, df3])

df = df.groupby('prolificID').filter(lambda x: len(x) >= 483)
df2 = df2.groupby('prolificID').filter(lambda x: len(x) >= 483)

df = pd.concat([df, df2])
# df = df.groupby('prolificID').filter(lambda x: len(x) <= 432)
# keep  prolificIDs that are more than 10 characters
df = df[df.prolificID.str.len() > 10]
# show prolificIDs

expName = ['exp1B',  'exp1', 'exp2', 'exp3']

df['opti_ss'] = ((df.m1 > df.m2) & (df.choice==1)) | ((df.m1 < df.m2) & (df.choice==2))
df['opti_ff'] = ((df.p1 > df.p2) & (df.choice==1)) | ((df.p1 < df.p2) & (df.choice==2))
df['opti_ev'] = ((df.ev1 > df.ev2) & (df.choice==1)) | ((df.ev1 < df.ev2) & (df.choice==2))

df = df[df.expName.isin(expName)]

# remove duplicates and only take first based on prolificID, t, session
df = df.drop_duplicates(subset=['prolificID', 't', 'session'])


map_prolificID = {pid: i for i, pid in enumerate(df.prolificID.unique())}
df['id'] = df.prolificID.map(map_prolificID)

print(f'N = {len(df.prolificID.unique())}')


N = 245


In [25]:

session = 3
df_ = df[(df.session==session)].groupby(['prolificID'], as_index=False).mean(numeric_only=True)

import scipy.stats as stats

df_['delta'] = df_.opti_ff - df_.opti_ss

df_['opti_ff'] = df_.opti_ff.astype(float)
df_['opti_ss'] = df_.opti_ss.astype(float)
df_['delta'] = df_.delta.astype(float)
df['opti_ff'] = df['opti_ff'].astype(float)
df['opti_ss'] = df['opti_ss'].astype(float)
         
df2 = df[(df.session==session) & (df.expName.isin(expName))]

def get_group2(row):
    opti_ff = df2[df2.prolificID==row.prolificID].opti_ff
    opti_ss = df2[df2.prolificID==row.prolificID].opti_ss
    ttest = stats.ttest_rel(opti_ff, opti_ss)
    p = ttest.pvalue
    t = ttest.statistic
    p_ss = stats.ttest_1samp(opti_ss, 0.5, alternative='greater').pvalue < 0.05 
    p_ff = stats.ttest_1samp(opti_ff, 0.5, alternative='greater').pvalue < 0.05

    if p > 0.05:
        if p_ss and p_ff:
            return 'balanced'
        if p_ss:
            return 'value'
        if p_ff:
            return 'perceptual'
        return 'random'
    
    if t > 0 and p_ff:
        return 'perceptual'
    if t < 0  and p_ss:
        return 'value'

    return 'random'

def get_group(row):
    opti_ff = df2[df2.prolificID==row.prolificID].opti_ff
    opti_ss = df2[df2.prolificID==row.prolificID].opti_ss
    p_ss = stats.ttest_1samp(opti_ss, 0.5, alternative='greater').pvalue < 0.05 
    p_ff = stats.ttest_1samp(opti_ff, 0.5, alternative='greater').pvalue < 0.05

    if p_ss and p_ff:
        return 'combined'
        
    
    if p_ff:
        return 'perceptual'
    if p_ss:
        return 'value'

    return 'random'
    # raise ValueError('unexpected value')

df_['group'] = df_.apply(get_group, axis=1)

df['group'] = df['prolificID'].map(df_.set_index('prolificID')['group'])

# df = df[df.group != 'random']
print(len(df.prolificID.unique()))

245


In [26]:
import pandas as pd
from pathlib import Path

# Get all CSV files in the data/external directory
data_dir = Path('../data/external/demographics/')
csv_files = sorted(data_dir.glob('prolific_demographic_export_*.csv'))
print(f'Found {len(csv_files)} CSV files to merge.')
# Read and concatenate all CSV files
dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    # Optional: add a column to track source file
    df['source_file'] = file.name
    dfs.append(df)

# Merge all dataframes
merged_df = pd.concat(dfs, ignore_index=True)

# Display info about the merged data
print(f"Total files merged: {len(csv_files)}")
print(f"Total rows: {len(merged_df)}")
print(f"Columns: {list(merged_df.columns)}")
print(f"\nFirst few rows:")
merged_df.head()

dm = merged_df.copy()
dm['prolificID'] = dm['Participant id']
print(f'N = {len(dm.prolificID.unique())}')

Found 11 CSV files to merge.
Total files merged: 11
Total rows: 665
Columns: ['Submission id', 'Participant id', 'Status', 'Custom study tncs accepted at', 'Started at', 'Completed at', 'Reviewed at', 'Archived at', 'Time taken', 'Completion code', 'Total approvals', 'Age', 'Sex', 'Ethnicity simplified', 'Country of birth', 'Country of residence', 'Nationality', 'Language', 'Student status', 'Employment status', 'source_file']

First few rows:
N = 628


In [27]:
dm.prolificID

0      5dd4767b32288646716dc98a
1      5cfb82a12bcca70018252f81
2      650ee274aa4447ac435b7830
3      63316daa57747115376760da
4      631c8e97db06f601f81bd82f
                 ...           
660    612bb0204ad8e7e8bed041eb
661    6637b21240220c2517fffa1a
662    6405feae7d6321bfaaf793e8
663    676276312d87ec9601be7d5c
664    675aaec58dfc729c338a52eb
Name: prolificID, Length: 665, dtype: object

In [29]:
dm2

,Submission id,Participant id,Status,Custom study tncs accepted at,Started at,Completed at,Reviewed at,Archived at,Time taken,Completion code,...,Age,Sex,Ethnicity simplified,Country of birth,Country of residence,Nationality,Language,Student status,Employment status,prolificID
0,66faddeb2b2c945ec978732c,66c34ed9630786ec5aece5bb,APPROVED,Not Applicable,2024-09-30T17:20:57.201000Z,2024-09-30T18:56:31.382000Z,2024-10-02T11:04:28.024000Z,2024-09-30T18:56:32.735272Z,5735.0,CJFYZJY7,...,22,Male,DATA_EXPIRED,United States,United States,DATA_EXPIRED,English,Yes,DATA_EXPIRED,66c34ed9630786ec5aece5bb
1,66faddffe98a90820081298f,66ba4608bc582e7db54007e5,APPROVED,Not Applicable,2024-09-30T17:21:12.799000Z,2024-09-30T18:08:34.998000Z,2024-10-02T11:04:28.470000Z,2024-09-30T18:08:35.686187Z,2843.0,CJFYZJY7,...,45,Male,Mixed,United States,United States,United States,English,No,DATA_EXPIRED,66ba4608bc582e7db54007e5
2,66fade0be0effae5cbd53823,6643670ad062f01ac1c4b788,APPROVED,Not Applicable,2024-09-30T17:21:21.540000Z,2024-09-30T18:31:52.243000Z,2024-10-02T11:04:28.772000Z,2024-09-30T18:32:33.892126Z,4231.0,CJFYZJY7,...,31,Male,White,United States,United States,United States,English,DATA_EXPIRED,Full-Time,6643670ad062f01ac1c4b788
3,66fade2cc27dc4d50e9eb08d,615e5b5f1e8fbfe2c701bc8e,APPROVED,Not Applicable,2024-09-30T17:21:57.457000Z,2024-09-30T18:03:52.223000Z,2024-10-02T11:04:32.791000Z,2024-09-30T18:07:04.882456Z,2515.0,CJFYZJY7,...,27,Female,Mixed,United States,United States,United States,English,Yes,DATA_EXPIRED,615e5b5f1e8fbfe2c701bc8e
4,66fade8e680750b80259118c,66ae3fb9006e6fc44a2b56df,APPROVED,Not Applicable,2024-09-30T17:23:49.039000Z,2024-09-30T18:07:57.874000Z,2024-10-02T11:04:29.071000Z,2024-09-30T18:07:58.577689Z,2649.0,CJFYZJY7,...,28,Male,White,United States,United States,United States,English,No,DATA_EXPIRED,66ae3fb9006e6fc44a2b56df
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,670024b6e5859ddb5bf04587,5eea537ffd89d01283ffac22,APPROVED,Not Applicable,2024-10-04T17:31:25.423000Z,2024-10-04T18:20:44.014000Z,2024-10-08T13:02:41.141000Z,2024-10-04T18:20:45.397410Z,2959.0,CJFYZJY7,...,30,Female,White,United States,United States,United States,English,DATA_EXPIRED,"Not in paid work (e.g. homemaker', 'retired or...",5eea537ffd89d01283ffac22
62,670024ce3f3b2dff8c238dfb,65c140b80efce0ed096216e3,APPROVED,Not Applicable,2024-10-04T17:24:54.634000Z,2024-10-04T18:47:53.625000Z,2024-10-08T13:02:41.503000Z,2024-10-04T18:50:23.855779Z,4979.0,CJFYZJY7,...,36,Female,Black,United States,United States,United States,English,DATA_EXPIRED,DATA_EXPIRED,65c140b80efce0ed096216e3
63,670024dce7ffbf2b4d8aa630,66c7abc47de7196e9cac1014,APPROVED,Not Applicable,2024-10-04T17:25:04.564000Z,2024-10-04T18:12:20.844000Z,2024-10-08T13:02:45.684000Z,2024-10-04T18:12:22.940726Z,2837.0,CJFYZJY7,...,35,Male,Asian,Bangladesh,United States,United States,English,No,Full-Time,66c7abc47de7196e9cac1014
64,670045c5f8dcf3f10939bb8f,661571d6cc16ee34676734df,APPROVED,Not Applicable,2024-10-04T19:45:16.495000Z,2024-10-04T20:24:15.362000Z,2024-10-08T13:02:41.851000Z,2024-10-04T20:25:03.307938Z,2339.0,CJFYZJY7,...,41,Female,Black,United States,United States,United States,English,No,DATA_EXPIRED,661571d6cc16ee34676734df


In [28]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# 1. Basic Statistics
print("="*50)
print("DEMOGRAPHICS SUMMARY")
print("="*50)
print(f"\nTotal Participants: {len(dm2)}")
print(f"\nAge Statistics:")
print(dm2['Age'].describe())

print(f"\nSex Distribution:")
print(dm2['Sex'].value_counts())
print(f"\n{dm2['Sex'].value_counts(normalize=True)*100}")

print(f"\nTop 10 Nationalities:")
print(dm2['Nationality'].value_counts().head(10))

print(f"\nExperiment Distribution:")
print(dm2['expName'].value_counts())

# for each experiment, print number of participants classified as random
print("\nRandom Choice Classification by Experiment:")
for exp in dm2['expName'].unique():
    pids_exp = dm2[dm2.expName == exp].prolificID.unique()
    n_random = df[(df.prolificID.isin(pids_exp)) & (df.group == 'random')].prolificID.nunique()
    n_total = len(pids_exp)
    print(f"{exp}: {n_random} out of {n_total} participants classified as random ({(n_random/n_total)*100:.2f}%)")  

# 2. Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age distribution
axes[0, 0].hist(dm2['Age'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].axvline(dm2['Age'].mean(), color='red', linestyle='--', label=f'Mean: {dm2["Age"].mean():.1f}')
axes[0, 0].legend()

# Sex distribution
sex_counts = dm2['Sex'].value_counts()
axes[0, 1].bar(sex_counts.index, sex_counts.values, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Sex')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Sex Distribution')
for i, v in enumerate(sex_counts.values):
    axes[0, 1].text(i, v + 0.5, str(v), ha='center', va='bottom')

# Top nationalities
top_nat = dm2['Nationality'].value_counts().head(10)
axes[1, 0].barh(range(len(top_nat)), top_nat.values, edgecolor='black', alpha=0.7)
axes[1, 0].set_yticks(range(len(top_nat)))
axes[1, 0].set_yticklabels(top_nat.index)
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_title('Top 10 Nationalities')
axes[1, 0].invert_yaxis()

# Experiment distribution
exp_counts = dm2['expName'].value_counts()
axes[1, 1].bar(range(len(exp_counts)), exp_counts.values, edgecolor='black', alpha=0.7)
axes[1, 1].set_xticks(range(len(exp_counts)))
axes[1, 1].set_xticklabels(exp_counts.index, rotation=45, ha='right')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Experiment Distribution')
for i, v in enumerate(exp_counts.values):
    axes[1, 1].text(i, v + 0.5, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

# 3. Age by Sex
print("\n" + "="*50)
print("AGE BY SEX")
print("="*50)
print(dm2.groupby('Sex')['Age'].describe())

# 4. Age by Experiment
print("\n" + "="*50)
print("AGE BY EXPERIMENT")
print("="*50)
print(dm2.groupby('expName')['Age'].describe())

# Optional: Additional boxplot for Age by Sex
plt.figure(figsize=(8, 6))
sns.boxplot(data=dm2, x='Sex', y='Age')
plt.title('Age Distribution by Sex')
plt.ylabel('Age')
plt.show()

DEMOGRAPHICS SUMMARY

Total Participants: 64

Age Statistics:
count     64
unique    29
top       31
freq       5
Name: Age, dtype: object

Sex Distribution:
Sex
Male               32
Female             31
CONSENT_REVOKED     1
Name: count, dtype: int64

Sex
Male               50.0000
Female             48.4375
CONSENT_REVOKED     1.5625
Name: proportion, dtype: float64

Top 10 Nationalities:
Nationality
United States          57
Nigeria                 2
DATA_EXPIRED            1
Spain                   1
Chile                   1
Trinidad and Tobago     1
CONSENT_REVOKED         1
Name: count, dtype: int64

Experiment Distribution:


KeyError: 'expName'